# Day 9 — Terminal and Shell for Engineers: Paths, Pipes, grep, Environment Variables, Exit Codes, Processes & Ports (2 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: become fast and safe in the terminal. You learn where you are in the file system and why paths break scripts, how programs talk through stdin, stdout and stderr, how pipes and grep answer questions about a log in seconds, how environment variables and PATH decide what runs, how exit codes tell cron and CI whether a job worked, and how to find and stop a process that is holding a port.

- Deliverable: day09/log_report.py, a CLI that reads a server log from stdin, prints an error summary and exits with a meaningful code; day09/drills.sh with six shell one-liners about the same log; and day09/dsa.py. The sample log day09/server.log is in the repo.

- Where this sits: Week 2, Phase 1 (Foundation), day 2 of 7. Tomorrow's pytest runs from the terminal, Week 3's FastAPI server listens on a port, and every Docker and cloud deploy in Phase 3 is configured with environment variables and judged by exit codes.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing every command in day09/ |
| Hands-on | 2.5h | Day 8 PRs if still open, drills.sh, log_report.py |
| DSA | 0.5h | Longest error streak, simplify a path |
| Revision | 0.5h | Quiz, Remember cells in the notebook, open items |

Open items first: the repo on GitHub still has only day01/ to day04/, and day04/ holds the starter TODOs. So Day 8's two pull requests (Days 5 and 6 into main) are not merged yet. If that is true on your machine too, spend the first 75 minutes of hands-on time on Day 8 Steps 1 to 3, then do today's task. The pull requests matter more than finishing every part of today: they put your real work where the weekly score and hiring managers can see it. Today's work goes in through a branch and a pull request as well (day09-log-report).

Mac note: your Mac runs zsh, and this session was run in bash on Linux. Every command here works the same in both. Where macOS prints something slightly different (an error message, an error number), the comment says so.

## Recap (2 minutes, from memory)

- What is a branch in Git, in one sentence? What moves when you commit?

- Your feature branch was pushed, then you rebased it onto main. The next push is rejected. Which command fixes it, and why that one rather than --force?

- What is the difference between git diff and git diff --staged?

Predict what each command prints, then run it in a throwaway repo (your git-demo folder from yesterday):

```bash
# Recap: predict the output of every line, then run it.
git switch -c recap-demo  # new branch from main, and switch to it
echo "a" > a.txt && git add a.txt && git commit -q -m "Add a.txt"
git switch main
git branch --merged  # branches whose commits are all inside main
git merge --ff-only recap-demo  # merge only if it can be a fast-forward
git branch --merged
git branch -d recap-demo  # -d refuses to delete unmerged work
```

Answers are on the last page of today's session.

## Concept 1: Paths and the file system

### Why it matters for an FDE

At a customer you will often work on a machine you have never seen: a jump host, a container, a VM in their cloud, reached over SSH with no VS Code and no mouse. The first questions are always where am I, what is here, and where are the logs. The most common reason a script works on your laptop and fails in cron or Docker is a relative path that only made sense from one folder.

### How it works

The file system is one tree. On macOS and Linux it starts at / (the root). Your home folder is /Users/rohit on a Mac (/home/rohit on Linux), and ~ is the shell's short name for it. Every terminal has a current working directory (cwd): the folder it is "in". pwd prints it, cd changes it, ls lists what is in it.

An absolute path starts with / and means the same thing from anywhere: /Users/rohit/fde-journey/day09/server.log. A relative path is looked up from the cwd: day09/server.log only works when the cwd is the repo root. Two special names help: . is the current folder and .. is its parent. It is the same rule as imports in a React project: ./Header and ../utils/api are relative to the file, / would be the root.

```bash
# Concept 1: moving around. Type each line and compare with the comments.
cd ~/fde-journey  # ~ = your home folder, so this works from anywhere
pwd  # print working directory
# -> /Users/rohit/fde-journey
ls  # list the cwd (hidden files are not shown)
# -> README.md  day04  day09
cd day09  # relative: day09 inside the cwd
pwd
# -> /Users/rohit/fde-journey/day09
ls ../day04  # .. = the parent folder, so this lists a sibling folder
# -> contact_store.py  dsa.py  main.py
cd ..  # up one level
pwd
# -> /Users/rohit/fde-journey
cd /Users/rohit/fde-journey/day09  # absolute: same result from anywhere
cd -  # jump back to the previous folder, and print it
# -> /Users/rohit/fde-journey
ls -a  # -a also shows names that start with a dot (hidden files)
# -> .  ..  .git  .gitignore  README.md  day04  day09
# FDE TIP: press Tab to complete names, and type cd day then Tab twice
# to see every match. It is faster and you never mistype a path.
# GOTCHA: names with spaces need quotes: cd "My Projects", or the shell
# splits them into two words.
```

Your shell's cwd and your Python script's cwd are the same thing: the folder you ran the command from, not the folder the .py file lives in. Day 4 showed Path(__file__).parent as the fix; here is the proof. Save this as day09/where_am_i.py and run it from two places:

In [ ]:
# Concept 1: the folder you run FROM vs the folder the script lives IN.
from pathlib import Path

# Path.cwd(): the working directory of the terminal that started us
print("cwd:   ", Path.cwd())
# __file__ is this script's own path; .resolve() makes it absolute
here = Path(__file__).resolve().parent
print("script:", here)

# A relative path is looked up from cwd, NOT from the script's folder
print("relative ok?", Path("server.log").exists())
# Anchored to the script's folder: works from anywhere
print("anchored ok?", (here / "server.log").exists())

# Run 1, from inside day09:   cd day09 && python3 where_am_i.py
# -> cwd:    /Users/rohit/fde-journey/day09
# -> script: /Users/rohit/fde-journey/day09
# -> relative ok? True
# -> anchored ok? True
# Run 2, from the repo root:  python3 day09/where_am_i.py
# -> cwd:    /Users/rohit/fde-journey
# -> script: /Users/rohit/fde-journey/day09
# -> relative ok? False      (it looked for fde-journey/server.log)
# -> anchored ok? True
# REMEMBER: relative paths follow the terminal, not the file.

### Real-world FDE example

A customer's ops team schedules your Contentstack export with cron. Cron starts jobs with the cwd set to the user's home folder, so open("config.json") fails at 2 a.m. even though it worked every time you ran it by hand from the project folder. The fix is to build every path from Path(__file__).parent or to take the path as an argument (--config /etc/cms-export/config.json), and to put the absolute path in the runbook.

### Common mistakes

- Running python3 day09/log_report.py from the repo root and then wondering why a relative path inside the script cannot find its file.

- Typing cd day09 when you are already inside day09. Run pwd before you guess.

- Forgetting hidden files: .env, .git and .gitignore only show with ls -a.

- Writing Windows-style paths with backslashes in code. Use pathlib and forward slashes; they work on every OS.

## Concept 2: stdin, stdout, stderr, redirection and pipes

### Why it matters for an FDE

Most of the tools you will glue together at a customer (curl, jq, psql, grep, your own scripts) speak through three text streams. If your script writes its result to stdout and its warnings to stderr, it can be chained with any other tool, saved to a file, or run by cron with errors mailed to someone. If it mixes them, every pipe that uses it gets corrupted.

### How it works

Every process starts with three open streams. You met two of them on Day 6 (print(..., file=sys.stderr)); today you control them from the shell.

| Stream | Number | Default | Python | Node equivalent |
|---|---|---|---|---|
| stdin | 0 | keyboard | sys.stdin | process.stdin |
| stdout | 1 | screen | print(), sys.stdout | console.log() |
| stderr | 2 | screen | print(..., file=sys.stderr) | console.error() |

Redirection points a stream at a file instead: > sends stdout to a file (emptying it first, like Day 4's mode "w"), >> appends, < feeds a file into stdin, 2> sends stderr to a file, and 2>&1 means "send stderr wherever stdout is going". /dev/null is a file that throws away everything written to it. A pipe | connects one program's stdout to the next program's stdin, so small tools combine into a pipeline, the same idea as chaining .filter().map() in JavaScript.

```bash
# Concept 2: redirection. Run inside day09/, where server.log lives.
echo "hello"  # stdout goes to the screen
# -> hello
echo "hello" > out.txt  # nothing printed: stdout went into the file
cat out.txt  # cat prints a file
# -> hello
echo "again" >> out.txt  # >> appends instead of emptying the file
cat out.txt
# -> hello
# -> again
ls missing.txt > out.txt  # the error still reaches the screen...
# -> ls: cannot access 'missing.txt': No such file or directory
cat out.txt  # ...because errors use stderr; > only moved stdout
# -> (nothing: > emptied the file and ls wrote nothing to stdout)
ls missing.txt 2> err.txt  # 2> moves stderr into a file
cat err.txt
# -> ls: cannot access 'missing.txt': No such file or directory
ls server.log missing.txt > all.txt 2>&1  # both streams, one file
# GOTCHA: order matters. "2>&1 > all.txt" sends errors to the screen.
wc -l < server.log  # < feeds the file to stdin: count lines
# -> 37
cat server.log | wc -l  # the same, through a pipe
# -> 37
# macOS prints the ls error as: ls: missing.txt: No such file or directory
```

A Python script joins a pipeline by reading sys.stdin, which is a file object exactly like the ones from Day 4: loop over it and you get one line at a time. Save this as day09/only.py:

In [ ]:
# Concept 2: a filter that reads stdin, writes stdout, reports on stderr.
import sys

# sys.argv: the words typed after "python3", like process.argv in Node
wanted = sys.argv[1] if len(sys.argv) > 1 else "ERROR"
kept = 0
for line in sys.stdin:  # one line per loop; works for huge inputs too
    if f" {wanted} " in line:  # spaces avoid matching "ERRORS" etc.
        sys.stdout.write(line)  # line already ends in "\n"
        kept += 1
# The count is a message for the human, not data: it goes to stderr
print(f"kept {kept} {wanted} lines", file=sys.stderr)

# $ python3 only.py WARN < server.log
# -> 2026-11-01T22:00:09Z WARN [sync] slow response status=200 took=3904ms
# -> ... (4 WARN lines)
# -> kept 4 WARN lines
# $ python3 only.py WARN < server.log > warn.txt
# -> kept 4 WARN lines      (stderr still on screen; data is in the file)
# $ python3 only.py ERROR < server.log 2>/dev/null | wc -l
# -> 9                      (wc counted data only, never the message)
# REMEMBER: data to stdout, messages to stderr.

### Real-world FDE example

A customer's Drupal-to-Contentstack migration produces a 2 GB JSON Lines export. You do not open it in an editor. You run head -n 3 export.jsonl to see the shape, wc -l export.jsonl to count entries, and python3 validate.py < export.jsonl > bad_rows.jsonl to stream it through your checker, with progress messages on stderr so the output file stays clean JSON.

### Common mistakes

- Using > when you meant >>, and wiping a log or a results file.

- sort file > file: the shell empties the file before sort reads it, so you lose the data. Write to a new file, then rename it.

- Printing progress or debug lines to stdout in a script whose output is piped somewhere. They end up inside the data.

- Calling sys.stdin.read() on a huge file: it loads everything into memory. Loop over the lines instead.

## Concept 3: grep and friends, answering questions about a log

### Why it matters for an FDE

"The webhook failed overnight" is one of the most common messages you will get from a customer. On their server there is no dashboard, only log files. Being able to answer "how many, since when, which part" in two minutes with grep, sort and uniq is what makes a customer trust you on the first call. It is also how you check your own script's output before you trust it.

### How it works

Each tool does one job on lines of text, and pipes combine them:

- grep PATTERN file prints the lines that contain the pattern. -c counts them, -n adds line numbers, -i ignores case, -v inverts (lines that do NOT match), -o prints only the matching part, -E allows extended patterns such as (429|503), -r searches every file in a folder, -q prints nothing and only sets the exit code.

- head -n 5 and tail -n 5 show the first or last lines; tail -f keeps printing new lines as a log grows (Ctrl+C stops it).

- wc -l counts lines. cut -d " " -f 3 keeps the third field, split on spaces.

- sort sorts lines, sort -rn sorts numbers from highest to lowest, and uniq -c collapses repeated neighbouring lines into one, with a count in front.

- less file opens a file you can scroll and search (/text to search, q to quit).

Run each line inside day09/ against today's server.log. Each log line has the shape timestamp LEVEL [component] message.

```bash
# Concept 3: questions about server.log, one pipeline each.
grep -c ERROR server.log  # how many ERROR lines?
# -> 9
grep -n "timed out" server.log  # which lines, with line numbers?
# -> 7:2026-11-01T22:00:11Z ERROR [sync] request timed out after 10s page=3
# -> 8:2026-11-01T22:00:21Z ERROR [sync] request timed out after 10s ...
grep -v -e INFO -e DEBUG server.log | head -3  # everything else, first 3
# -> 2026-11-01T22:00:09Z WARN [sync] slow response status=200 took=3904ms
# -> 2026-11-01T22:00:11Z ERROR [sync] request timed out after 10s page=3
# -> 2026-11-01T22:00:21Z ERROR [sync] request timed out after 10s ...
grep -E "status=(429|5[0-9]{2})" server.log  # rate limits and 5xx
# -> 2026-11-01T22:15:01Z ERROR [webhook] downstream search API status=503
# -> 2026-11-01T22:15:03Z ERROR [webhook] downstream search API ...
# -> 2026-11-01T22:40:07Z ERROR [sync] status=429 rate limited, waiting 2s
# Which component has the most errors? The classic counting pipeline:
grep ERROR server.log | cut -d " " -f 3 | sort | uniq -c | sort -rn
# ->       4 [webhook]
# ->       3 [sync]
# ->       2 [export]
grep -o "status=[0-9]*" server.log | sort | uniq -c | sort -rn
# ->       8 status=200
# ->       2 status=503
# ->       2 status=401
# ->       1 status=429
# (ties such as 503 and 401 may print in a different order on a Mac)
```

The counting pipeline is the shell version of Day 3's counting dict: cut picks the key, sort puts equal keys next to each other, uniq -c counts each run, and sort -rn ranks them. Why the first sort? Because uniq only merges neighbours. Without it you get this:

```bash
# Concept 3: uniq without sort counts runs, not totals.
grep ERROR server.log | cut -d " " -f 3 | uniq -c
# ->       2 [sync]
# ->       1 [webhook]
# ->       2 [export]
# ->       2 [webhook]
# ->       1 [sync]
# ->       1 [webhook]
# Real data is messy: count field 2 (the level) over EVERY line.
cut -d " " -f 2 server.log | sort | uniq -c
# ->       1               (line 17 is blank)
# ->       2 DEBUG
# ->       1 EROR          (a typo from an old logger: not a real level)
# ->       9 ERROR
# ->      19 INFO
# ->       4 WARN
# ->       1 log           (line 23: "--- log rotated by logrotate ---")
# FDE TIP: run this "count the field" check on any new log before you
# write a parser for it. It shows the junk lines your code must survive.
```

### Real-world FDE example

A customer says some blog posts never reached their search index yesterday. On the integration server you run grep -c "status=503" /var/log/cms-sync/*.log to see which day files have failures, then grep 503 sync-2026-11-01.log | head to read the first few, then the counting pipeline on the component field. In five minutes you can say: "the search API returned 503 for 14 minutes from 22:15; 37 entries were affected; here is the list". That sentence is what a customer wants.

### Common mistakes

- uniq -c without sort first, which counts runs instead of totals.

- Forgetting quotes around a pattern with spaces or special characters: grep timed out server.log searches for "timed" in a file called "out".

- Trusting grep -i error to find all problems. It misses "EROR", "failed" and "status=503". Look at the data first.

- Running cat file | grep x everywhere. It works, but grep x file is shorter and shows the file name when you search several files.

## Concept 4: Environment variables and PATH

### Why it matters for an FDE

Every serious deployment is configured through environment variables: API keys, the CMS environment (staging or production), log levels, database URLs. Docker, Kubernetes, GitHub Actions, AWS and Vercel all inject them the same way. And PATH, one special variable, decides which python3 runs when you type it, which is behind half of all "it works on my machine" bugs.

### How it works

Each process has its own set of environment variables, a dict of strings (Python's os.environ, Node's process.env). When a process starts another one, the child gets a copy of the parent's exported variables. The copy goes one way: a child can never change its parent's variables, which is why a Python script cannot set a variable in your terminal.

In the shell, NAME=value (no spaces around =) creates a shell variable that only the shell sees. export NAME puts it in the environment, so programs you start get it. NAME=value command sets it for that one command only. $NAME or "${NAME}" reads it inside double quotes; single quotes keep the text exactly as typed.

```bash
# Concept 4: shell variables, exported variables, one-shot variables.
echo "$HOME"  # a variable every shell has
# -> /Users/rohit
STACK=blog-prod  # GOTCHA: no spaces around =, or the shell runs "STACK"
echo "$STACK"
# -> blog-prod
python3 -c 'import os; print(os.environ.get("STACK"))'
# -> None          (a plain shell variable is not passed to children)
export STACK  # now it is in the environment: children get a copy
python3 -c 'import os; print(os.environ.get("STACK"))'
# -> blog-prod
LOG_LEVEL=DEBUG python3 -c 'import os; print(os.environ["LOG_LEVEL"])'
# -> DEBUG         (set for that one command only)
echo "${LOG_LEVEL:-not set}"  # :- gives a default when it is not set
# -> not set
echo 'single: $STACK'  # single quotes: no variable expansion
# -> single: $STACK
echo "double: $STACK"  # double quotes: expanded
# -> double: blog-prod
python3 -c 'import os; os.environ["STACK"] = "changed"'
echo "$STACK"  # the child changed ITS copy, never the parent's
# -> blog-prod
# FDE TIP: env | sort lists everything; env | grep CS_ finds your keys.
```

PATH is a list of folders separated by colons. When you type python3, the shell checks each folder in order and runs the first python3 it finds. which python3 shows which one that is. Activating a virtual environment (Day 6) does nothing magic: it puts the venv's bin folder at the front of PATH.

```bash
# Concept 4: PATH decides which program runs. Run from the repo root.
echo "$PATH" | tr ":" "\n"  # tr swaps each : for a new line
# -> /usr/local/bin
# -> /usr/bin
# -> /bin
# (your list is longer; with Homebrew it starts with /opt/homebrew/bin)
which python3  # the first python3 found on PATH
# -> /usr/local/bin/python3
source .venv/bin/activate  # Day 6: activate the project's venv
echo "$PATH" | tr ":" "\n" | head -2
# -> /Users/rohit/fde-journey/.venv/bin     (added at the front)
# -> /usr/local/bin
which python3
# -> /Users/rohit/fde-journey/.venv/bin/python3
deactivate  # removes it from PATH again
which python3
# -> /usr/local/bin/python3
# REMEMBER: "which X" first, whenever the wrong version seems to run.
```

To keep a variable across terminal sessions, add export NAME=value to ~/.zshrc and open a new terminal. Do that for settings such as EDITOR, never for customer secrets: those belong in a project .env file (ignored by Git, Day 8) or a secrets manager. A .env file is not loaded by Python or the shell on its own; a library such as python-dotenv or your deployment platform reads it.

### Real-world FDE example

You deploy the same CMS search connector to a customer's staging and production. The code is identical; only CS_ENVIRONMENT, CS_DELIVERY_TOKEN and SEARCH_API_URL differ, and their platform team sets them in Kubernetes. Your code reads os.environ, fails fast with a clear message when one is missing (Concept 5 shows how), and never has a key in the repo. This is the twelve-factor app rule "config in the environment", and customers' security teams check for it.

### Common mistakes

- Spaces around = (KEY = abc), which runs a command called KEY.

- Setting a variable without export and wondering why Python sees None.

- Running export CS_API_KEY=... in a terminal you then share on a call, or leaving it in shell history. Prefer .env files and read -s for typing secrets.

- Installing a package with one pip and running another python3. Check which python3 and which pip, or always use python3 -m pip.

## Concept 5: Exit codes, && and ||

### Why it matters for an FDE

Machines do not read your output; they read your exit code. Cron, CI pipelines, Docker health checks, Kubernetes jobs and Airflow tasks all decide "did it work?" from one number. A script that prints "Error: export failed" but exits with 0 makes the nightly job look green while the customer's data is stale, which is worse than a crash.

### How it works

Every command ends with an exit code from 0 to 255. 0 means success; anything else means failure, and the number can say what kind. The shell stores the last one in $?.

| Code | Usual meaning | Where you see it |
|---|---|---|
| 0 | success | every command that worked |
| 1 | general failure | uncaught Python exception, grep found nothing |
| 2 | wrong usage | argparse errors, bad options |
| 127 | command not found | a typo, or a tool not on PATH |
| 130 | stopped with Ctrl+C | 128 + signal 2 (SIGINT) |

a && b runs b only if a succeeded; a || b runs b only if a failed. That is how deployment scripts stop at the first broken step. In a pipeline, $? is the exit code of the last command only, unless you turn on set -o pipefail.

```bash
# Concept 5: read $? right after the command you care about.
ls server.log; echo "exit: $?"
# -> server.log
# -> exit: 0
ls missing.txt 2>/dev/null; echo "exit: $?"
# -> exit: 2          (macOS ls returns 1: any non-zero means failure)
grep -q "status=500" server.log; echo "exit: $?"
# -> exit: 1          (GOTCHA: grep returns 1 when nothing matches)
nosuchtool 2>/dev/null; echo "exit: $?"
# -> exit: 127        (command not found)
python3 -c 'import sys; sys.exit(3)'; echo "exit: $?"
# -> exit: 3
python3 -c 'raise ValueError("bad row")' 2>/dev/null; echo "exit: $?"
# -> exit: 1          (an uncaught exception always exits with 1)
grep -q ERROR server.log && echo "errors found, alert"
# -> errors found, alert
grep -q "status=500" server.log || echo "no 500s"
# -> no 500s
cat missing.txt 2>/dev/null | wc -l; echo "exit: $?"
# -> 0
# -> exit: 0          (only wc's code: the failed cat is hidden)
set -o pipefail  # now a pipeline fails if ANY part fails
cat missing.txt 2>/dev/null | wc -l; echo "exit: $?"
# -> 0
# -> exit: 1
```

In Python, the pattern every CLI in this course uses is a main() that returns an int, passed to sys.exit(). Here it powers a preflight check you would run before deploying, combining today's environment variables and exit codes. Save it as day09/preflight.py:

In [ ]:
# Concept 5: a preflight check that fails fast, with a useful exit code.
import os
import sys

# The settings this deployment cannot run without
REQUIRED = ["CS_API_KEY", "CS_DELIVERY_TOKEN", "CS_ENVIRONMENT"]

def main() -> int:
    """Return 0 if every required variable is set, else 1."""
    # .get() returns None for a missing variable; "" also counts as missing
    missing = [name for name in REQUIRED if not os.environ.get(name)]
    if missing:
        print("missing: " + ", ".join(missing), file=sys.stderr)
        return 1  # non-zero: the caller (shell, CI, cron) sees a failure
    print("config ok for", os.environ["CS_ENVIRONMENT"])
    return 0

if __name__ == "__main__":
    sys.exit(main())  # the int from main() becomes the exit code

# $ python3 preflight.py; echo "exit: $?"
# -> missing: CS_API_KEY, CS_DELIVERY_TOKEN, CS_ENVIRONMENT
# -> exit: 1
# $ CS_API_KEY=a CS_DELIVERY_TOKEN=b CS_ENVIRONMENT=staging \
#     python3 preflight.py && echo "deploying"
# -> config ok for staging
# -> deploying
# REMEMBER: main() returns the code; sys.exit(main()) hands it over.

The backslash at the end of a shell line continues the command on the next line; it is only there to keep the line short.

### Real-world FDE example

A customer's GitHub Actions workflow runs python3 preflight.py && python3 sync.py && python3 log_report.py < sync.log. If the delivery token was rotated and not updated, preflight exits 1, the workflow stops before touching any content, and the team gets a red build with "missing: CS_DELIVERY_TOKEN" as the message. No half-synced site, no guessing.

### Common mistakes

- Catching every exception, printing it, and finishing normally, so the script exits 0 after a failure.

- Reading $? after another command (even echo) has already run and replaced it.

- Treating grep's exit code 1 (no match) as a crash in a script. It is often the good news.

- Calling sys.exit() deep inside helper functions. Raise an exception or return a value, and decide the exit code once, in main().

## Concept 6: Processes, signals and ports

### Why it matters for an FDE

Next week you will run a FastAPI server with uvicorn, and in Phase 3 you will run several services in Docker. The first time something goes wrong you will see "Address already in use", a server that will not stop, or a job that hangs. Knowing how to find a process, stop it properly and see which port it holds saves hours, especially on a customer's machine where you cannot simply restart everything.

### How it works

A process is a running program. Each one has a number, its PID. ps lists processes, $! is the PID of the last background job, and a command ending in & runs in the background so the terminal stays free (jobs lists them, fg brings one back). You stop a process by sending it a signal: Ctrl+C sends SIGINT ("please stop"), kill PID sends SIGTERM ("please shut down"), and kill -9 PID sends SIGKILL, which ends it at once with no chance to clean up.

A port is a numbered door on a machine. A server listens on a port, and only one process can listen on a given port at a time. localhost (127.0.0.1) means this machine only; a server bound to 0.0.0.0 accepts connections from other machines too. You know this from React: npm run dev listens on port 3000 or 5173, and when the port is taken it asks to use another one. Python's servers just fail.

```bash
# Concept 6: start a server, see the port, hit the conflict, stop it.
python3 -m http.server 8000 > server_out.txt 2>&1 &  # & = background
echo "started PID $!"  # $! = PID of the job we just started
# -> started PID 533         (your number will differ)
curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8000/server.log
# -> 200                     (the server answered: file found)
python3 -m http.server 8000  # a second server on the same port...
# -> OSError: [Errno 98] Address already in use   (macOS: Errno 48)
lsof -nP -iTCP:8000 -sTCP:LISTEN  # who is listening on port 8000?
# -> COMMAND PID USER  FD TYPE DEVICE SIZE/OFF NODE NAME
# -> python3 533 root  3u IPv4 2333   0t0      TCP  *:8000 (LISTEN)
# (spacing trimmed; you will see your own PID and your user name)
kill %1  # SIGTERM to background job 1 (or: kill 533)
curl -s http://localhost:8000/; echo "curl exit: $?"
# -> curl exit: 7            (7 = could not connect: nobody listening)
# GOTCHA: kill -9 only when a plain kill fails. -9 gives the process no
# chance to flush files or finish a request, so it can leave half-written
# output behind (Day 4's atomic save exists because of this).
```

A long-running Python script should stop cleanly on Ctrl+C. Python turns SIGINT into a KeyboardInterrupt exception, so you handle it with Day 4's try/except. Save as day09/worker.py:

In [ ]:
# Concept 6: a long-running worker that stops cleanly on Ctrl+C.
import os
import sys
import time

def main() -> int:
    """Tick once a second; on Ctrl+C, clean up and exit with 130."""
    # flush=True prints at once, even when stdout is a pipe or a file
    print(f"worker PID {os.getpid()} started", flush=True)
    try:
        for tick in range(1, 6):
            print(f"tick {tick}", flush=True)
            time.sleep(1)  # pretend to sync one page of entries
    except KeyboardInterrupt:  # Ctrl+C arrives as this exception
        print("Ctrl+C received, saving progress and stopping",
              file=sys.stderr)
        return 130  # 128 + 2 (SIGINT): the standard code for Ctrl+C
    print("all ticks done")
    return 0

if __name__ == "__main__":
    sys.exit(main())

# $ python3 worker.py      (press Ctrl+C after about 3 seconds)
# -> worker PID 547 started
# -> tick 1
# -> tick 2
# -> tick 3
# -> Ctrl+C received, saving progress and stopping
# $ echo $?
# -> 130

### Real-world FDE example

During a live demo to a customer, your FastAPI app will not start: "Address already in use" on port 8000. Yesterday's server is still running in a forgotten terminal tab. lsof -nP -iTCP:8000 -sTCP:LISTEN shows its PID, kill stops it, and the demo continues within thirty seconds. Without these commands, the usual reaction is restarting the laptop in front of the customer.

### Common mistakes

- Closing the terminal window and assuming the server stopped. A background job or another tab may still hold the port.

- Reaching for kill -9 first, and losing data a clean shutdown would have saved.

- Binding a server to localhost inside Docker and wondering why it cannot be reached from outside (Phase 3 covers this).

- Forgetting flush=True (or python3 -u) and seeing no output from a background job until it ends.

## Check your understanding (predict first, then run)

Run these inside day09/. Write each prediction down before you press Enter.

```bash
# Check 1: what does cat print?
echo "one" > c.txt; echo "two" > c.txt; cat c.txt
# Check 2: what number is printed?
grep ERROR server.log | grep -c sync
# Check 3: what is printed, and what is the exit code?
grep "status=999" server.log; echo "exit: $?"
# Check 4: two lines. What are they?
X=5 bash -c 'echo "inside: $X"'; echo "after: ${X:-unset}"
# Check 5: which word is on the screen, and which is in o.txt?
python3 -c 'import sys; print("out"); print("err", file=sys.stderr)' \
    > o.txt
cat o.txt
```

- Check 1: what does cat print, and why?

- Check 2: what number, and what does each half of the pipe do?

- Check 3: what is printed and what is the exit code? Is that an error?

- Check 4: what are the two lines, and why is the second one so?

- Check 5: which word appears before cat runs, and which does cat print?

- Without running it: why is sort server.log > server.log a bad idea?

## Hands-on task (about 2.5 hours): day09/log_report.py

The scenario: a customer runs a nightly Contentstack sync, a webhook listener and a CSV export on one server, all writing to one log. Their on-call engineer wants a command that summarises last night's log and fails loudly when there were errors, so their cron can send an alert. day09/server.log is a real-shaped sample: 37 lines, including a blank line, a log-rotation marker and a line with a mistyped level.

Before you start: git switch main && git pull, then git switch -c day09-log-report. The starters are in day09/: drills.sh, log_report.py, dsa.py and server.log.

Part A, drills.sh (30 min). Six questions about server.log, one shell one-liner each, written into the TODO lines of drills.sh. Run it with bash drills.sh from inside day09/. The answers your commands print are in the file's comments, so you can check yourself; the commands are yours to find.

Part B, log_report.py (100 min). Complete the four functions in the starter: parse_line, summarize, format_report and main. The output for the sample log must look like this:

```bash
# Part B: the target. Run from inside day09/.
python3 log_report.py < server.log; echo "exit: $?"
# stderr (the warnings come first):
# -> warning: line 23: malformed, skipped
# -> warning: line 30: malformed, skipped
# stdout:
# -> Log report
# ->   lines read: 37
# ->   parsed:     34
# ->   skipped:    3
# -> By level:
# ->   ERROR  9
# ->   WARN   4
# ->   INFO   19
# ->   DEBUG  2
# -> Errors by component (top 3):
# ->   webhook  4
# ->   sync     3
# ->   export   2
# -> First error: 2026-11-01T22:00:11Z
# -> Last error:  2026-11-01T22:45:00Z
# -> exit: 1
```

Part C, use it like a real tool (20 min). Prove it works in pipelines: run the acceptance checks below, then commit in at least three commits (drills; parser and summary; report and exit codes), push and open a pull request with What, Why, How to test and Open items.

Acceptance criteria:

- bash drills.sh prints six answers that match the expected values in the file's comments.

- python3 log_report.py < server.log prints exactly the report above on stdout, and the two warnings on stderr only: python3 log_report.py < server.log 2>/dev/null shows no warning lines.

- The blank line is skipped silently; malformed lines (wrong number of fields, unknown level, component without brackets) are skipped with a warning that names the line number. The script never crashes on them.

- Exit code 1 when there is at least one ERROR, 0 when there is none: grep -v ERROR server.log | python3 log_report.py; echo $? prints 0.

- python3 log_report.py --top 2 < server.log shows two components; --top 0 and --top abc print a usage error and exit with 2.

- Run with no input (just python3 log_report.py in a terminal), it does not hang: it prints a one-line usage hint to stderr and exits 2.

- cat server.log | python3 log_report.py and python3 day09/log_report.py < day09/server.log (from the repo root) both work.

- Pressing Ctrl+C while it waits for input exits with 130 and no traceback.

- The pull request is open (or merged) with at least three commits, and the Day 8 pull requests are merged before it.

Hints (no solutions, try first):

- line.split(maxsplit=3) splits on any run of spaces, at most three times, so the message keeps its own spaces. Count the parts before you unpack them.

- enumerate(sys.stdin, start=1) gives you (line_number, line) pairs, which is what the warnings need.

- The top-N list is Day 3's counting dict plus Day 4's sorted(..., key=...). Break ties alphabetically so the output is stable.

- sys.stdin.isatty() is True when nobody piped anything in and the script would wait for the keyboard.

- For --top 0, argparse cannot know that 0 is wrong. After parse_args(), call parser.error("..."): it prints usage and exits with 2 for you.

- Test exit codes by hand first: python3 log_report.py < server.log > /dev/null 2>&1; echo $? should print 1.

- Stuck on output alignment? f-strings pad text: f"{name:<8}" makes name at least 8 characters wide, left-aligned.

## DSA practice (30 min)

Both problems are in day09/dsa.py, with signatures, examples and hints.

- Longest error streak. longest_streak(levels, target) returns the length of the longest run of target in a row. longest_streak(["INFO", "ERROR", "ERROR", "WARN", "ERROR"], "ERROR") gives 2. Hint: walk the list once, keeping the current run length and the best so far; decide what happens to the current run when a different value appears. It is the check an alerting rule does for "5 errors in a row".

- Simplify a path. simplify_path("/home/rohit/../rohit/./fde//day09/") gives "/home/rohit/fde/day09", the same answer cd and Path.resolve() would give. Hint: split on /, ignore empty parts and ., and use a list as a stack (Day 4's brackets problem): a folder name is pushed, .. pops, but never pops past the root.

## Quiz (3 quick questions)

- A cron job runs python3 export.py > export.log every night, and the file has no error messages even on nights when the export failed. Give two reasons this can happen and the fix for each.

- You pip install requests and then python3 cms_export.py still fails with ModuleNotFoundError: requests. Which two commands do you run first, and what are you looking for?

- Your FastAPI server will not start: "Address already in use". Describe the three steps to fix it safely, with the commands.

## What you achieved today

You can now find your way around any Unix machine, explain why a relative path breaks under cron, route stdin, stdout and stderr with redirection and pipes, answer real questions about a log with grep, cut, sort and uniq, configure programs with environment variables, debug PATH, read and set exit codes, and find and stop a process holding a port. Your log_report.py is a tool a customer's on-call engineer could put in cron tonight.

Tomorrow (Day 10): testing with pytest (asserts, fixtures, parametrize, testing errors and faking HTTP), writing green test suites for Day 4's contact_store.py and Day 6's cms_export.py.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 9 review": your recap and check-your-understanding predictions next to the real outputs, drills.sh, log_report.py with the output of python3 log_report.py < server.log; echo $?, dsa.py, the link to today's pull request (and the two Day 8 ones), and your quiz answers. Say which concept felt least natural; that is where tomorrow's recap will start.

## Recap answers

*Your answers:*
